# DETR — End-to-End Object Detection with Transformers (2020)

_Papers · Computer Vision_

**Cast detection as direct set prediction: a CNN + transformer outputs a fixed set of boxes, matched to truth one-to-one by a Hungarian loss — no anchors, no NMS.**

---

This notebook is a practice scaffold for the **DETR — End-to-End Object Detection with Transformers (2020)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

np.set_printoptions(precision=3, suppress=True)

# --- 1. The worked example: N=3 predictions, M=2 ground-truth objects. -------------
# Predicted class probabilities over [A, B, no-object] and boxes (cx, cy, w, h).
probs = np.array([[0.7, 0.2, 0.1],
                  [0.1, 0.8, 0.1],
                  [0.3, 0.3, 0.4]])
pred_boxes = np.array([[0.20, 0.20, 0.10, 0.10],
                       [0.62, 0.55, 0.20, 0.20],
                       [0.50, 0.50, 0.30, 0.30]])
gt_class = [0, 1]                                  # object 0 is class A, object 1 is class B
gt_boxes = np.array([[0.25, 0.25, 0.12, 0.12],
                     [0.60, 0.60, 0.18, 0.22]])
LAM_L1 = 1.0

def match_cost(probs, pred_boxes, gt_class, gt_boxes, lam_l1=1.0):
    N, M = len(pred_boxes), len(gt_boxes)
    C = np.zeros((N, M))
    for i in range(N):
        for j in range(M):
            cls = gt_class[j]
            l1 = np.abs(pred_boxes[i] - gt_boxes[j]).sum()   # box loss (toy: L1 only)
            C[i, j] = -probs[i, cls] + lam_l1 * l1           # -prob(true class) + box loss
    return C                                                  # NOTE: probability, not -log

C = match_cost(probs, pred_boxes, gt_class, gt_boxes, LAM_L1)
print("Cost matrix C (rows=preds, cols=gt A,B):\n", C)
print("C[0,0] = -0.70 + 0.14 =", round(C[0, 0], 3))          # -0.56, the worked value

# Hungarian algorithm = optimal ONE-TO-ONE assignment (Eq. 1).
rows, cols = linear_sum_assignment(C)
print("\nHungarian matching pred->gt:", list(zip(rows.tolist(), cols.tolist())))  # [(0,0),(1,1)]
print("total matching cost:", round(C[rows, cols].sum(), 3))                       # -1.25
unmatched = sorted(set(range(len(pred_boxes))) - set(rows.tolist()))
print("unmatched predictions (-> no-object):", unmatched)                          # [2]


# --- 2. The Hungarian set loss on the matched pairs (Eq. 2, toy). ------------------
EPS = 1e-9
cls_loss = 0.0
box_loss = 0.0
for i, j in zip(rows, cols):
    cls_loss += -np.log(probs[i, gt_class[j]] + EPS)                 # -log p(true class)
    box_loss += np.abs(pred_boxes[i] - gt_boxes[j]).sum()           # matched box loss
# unmatched predictions are pushed toward 'no-object' (index 2), down-weighted x10 in the paper
no_obj = sum(-0.1 * np.log(probs[i, 2] + EPS) for i in unmatched)
total = cls_loss + box_loss + no_obj
print("\nset loss  -> class:", round(cls_loss, 3),
      " box:", round(box_loss, 3), " no-obj(x0.1):", round(no_obj, 3),
      " TOTAL:", round(total, 3))


# --- 3. ABLATION: Hungarian (one-to-one) vs greedy (collides) --------------------
# A cost matrix where prediction 0 is the cheapest match for BOTH objects.
C2 = np.array([[-0.9, -0.8],
               [ 0.2,  0.1],
               [ 0.3,  0.3]])
h_rows, h_cols = linear_sum_assignment(C2)
greedy = [(int(np.argmin(C2[:, j])), j) for j in range(C2.shape[1])]  # each obj picks its best pred
print("\nABLATION on collision matrix C2:\n", C2)
print("Hungarian:", list(zip(h_rows.tolist(), h_cols.tolist())),
      "-> distinct preds:", len(set(h_rows.tolist())))                # 2 distinct -> one-to-one
print("Greedy   :", greedy,
      "-> distinct preds:", len(set(p for p, _ in greedy)))           # 1 distinct -> COLLISION
print("Greedy reuses prediction 0 for both objects -> a DUPLICATE that NMS would delete.")
print("Hungarian forbids reuse -> no duplicate, no NMS needed. (Toy run, not the paper's COCO AP.)")

## Visualize the data & results

_When one prediction is the best match for two objects, does the assignment stay one-to-one (Hungarian) or collide (greedy)?_

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

# Collision cost matrix: prediction 0 is the cheapest match for BOTH objects.
C2 = np.array([[-0.9, -0.8],
               [ 0.2,  0.1],
               [ 0.3,  0.3]])

h_rows, h_cols = linear_sum_assignment(C2)                 # one-to-one
greedy = [int(np.argmin(C2[:, j])) for j in range(C2.shape[1])]  # per-object argmin

hungarian_distinct = len(set(h_rows.tolist()))            # 2 -> no duplicate
greedy_distinct    = len(set(greedy))                     # 1 -> collision/duplicate
print("Hungarian distinct preds:", hungarian_distinct)    # 2
print("Greedy    distinct preds:", greedy_distinct)       # 1
# Greedy reuses prediction 0 for both objects (a duplicate NMS must delete);
# Hungarian's one-to-one bijection prevents it -> DETR needs no NMS.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The one-to-one ablation. Build a $3\times2$ cost matrix in which prediction $0$ is the cheapest
            (best) match for both objects. Run Hungarian and greedy. Which one assigns prediction $0$ to
            both objects, and what real-world artifact is that?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use $C = \begin{bmatrix} -0.9 & -0.8 \\ 0.2 & 0.1 \\ 0.3 & 0.3 \end{bmatrix}$: prediction $0$ has the lowest cost in both columns. — _This is the collision case: a single prediction is the best fit for two distinct objects._
- Greedy: per object, take $\arg\min$ over rows independently &rarr; object 0 picks row 0, object 1 also picks row 0. Both objects grabbed prediction $0$. — _Greedy lets objects choose without coordinating, so the same prediction can be reused &mdash; a duplicate._
- Hungarian: minimize total cost with distinct rows &rarr; $\{0\!\to\!\text{obj}0,\,1\!\to\!\text{obj}1\}$, total $-0.9 + 0.1 = -0.8$. — _The one-to-one constraint forbids reusing row 0, so it pays a bit more on object 1 to keep a valid bijection._

**Answer:** Greedy assigns prediction $0$ to both objects &mdash; a collision. In a detector
                 that means two boxes pointing at the same object: exactly the duplicate that NMS exists to
                 delete (and object 1 is left explained by a reused box). Hungarian forbids this: it returns
                 $\{0\!\to\!\text{obj}0,\ 1\!\to\!\text{obj}1\}$ at total cost $-0.8$, a valid one-to-one
                 assignment. Because training always uses such a bijection, DETR learns to fire once per object and
                 needs no NMS. The CODEVIZ panel shows this contrast.

</details>

**Problem 2.** Recompute one matching-cost entry by hand. Using the worked example's prediction $1$ and object $B$
            (object 1), with $\lambda_{L1}=1$, compute $C[1,1]$. Prediction $1$: $\hat p=[0.1,0.8,0.1]$,
            $\hat b=(.62,.55,.20,.20)$; object $B$: class $=B$, $b^{gt}=(.60,.60,.18,.22)$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Class term: object $B$'s class is $B$ (index 1), so $-\hat p_1(B) = -0.8$. — _The matching cost rewards probability on the true class &mdash; negated, so more probability lowers cost._
- Box L1: $|.62-.60| + |.55-.60| + |.20-.18| + |.20-.22| = 0.02+0.05+0.02+0.02 = 0.11$. — _L1 is the sum of absolute differences over the 4 box coordinates $(c_x,c_y,w,h)$._
- Sum: $C[1,1] = -0.8 + 0.11 = -0.69$. — _Matching cost = (negative class probability) + (box loss)._

**Answer:** $C[1,1] = -0.8 + 0.11 = \mathbf{-0.69}$ &mdash; the most negative (best) entry in the matrix,
                 which is why prediction $1$ matches object $B$. This is exactly the value in the worked-example
                 matrix, and the notebook reproduces it.

</details>

**Problem 3.** Why does DETR use the raw probability $\hat p(c_i)$ in the matching cost (Eq. 1) but the log
            $-\log\hat p(c_i)$ in the training loss (Eq. 2)? What goes wrong if you swap them?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- In matching you only need to rank assignments. A probability is in $[0,1]$, the same scale as the box loss, so neither term dominates the ranking. — _The paper: "This makes the class prediction term commensurable to $\mathcal{L}_{\text{box}}$" (&sect;3.1)._
- In the loss you optimize parameters; $-\log\hat p$ gives the standard cross-entropy gradient and strongly penalizes near-zero probability on the true class. — _Log-likelihood is the well-behaved objective for gradient training; a bare probability has weak gradients near 0._
- If you put $-\log\hat p$ in the matching cost, a single very-confident-wrong prediction can produce a huge term that dominates the box distances and distorts the assignment. — _An unbounded log term is not "commensurable" with the bounded box loss &mdash; the very thing the paper avoided._

**Answer:** Matching only ranks, so DETR uses the bounded probability $\hat p(c_i)\in[0,1]$ to keep it
                 "commensurable" with the box loss (&sect;3.1); training optimizes, so it uses $-\log\hat p$ for
                 well-behaved cross-entropy gradients. Swapping them lets an unbounded log term dominate the
                 assignment and "we observed better empirical performances" with the probability form (&sect;3.1).

</details>